# Benchmark DESPUÉS del fine-tuning (Qwen3-4B + LoRA)

Mide el mismo extractor **con** el LoRA `checkpoint-300` sobre los mismos dos benchmarks, y al final lo compara con el antes.

| | antes | después (este) |
|---|---|---|
| Modelo | Qwen3-4B-Instruct-2507 | Qwen3-4B-Instruct-2507 + LoRA `checkpoint-300` |
| Prompt | `PROMPT_SEMANTICO` (6 campos) | el mismo |
| Fecha | `resolvedor_temporal` | el mismo |
| Criterio | `evaluar.py` del D1 | el mismo |

**Pasos:** GPU **T4** → correr las celdas en orden. Te pide dos veces archivos:
1. Celda del adapter: `adapter_config.json` y `adapter_model.safetensors`.
2. Última celda: `resultados_antes.zip` (el que bajó `benchmark_antes.ipynb`).

Tarda ~15 min. Se descargan `resultados_despues.zip` y `comparacion.zip`.

In [ ]:
# @title 1. Instalar dependencias (mismas versiones con las que se entrenó el adapter)
!nvidia-smi -L
%pip install -q "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0"
%pip uninstall -y -q torchao
import torch
assert torch.cuda.is_available(), "Sin GPU: cambia el entorno de ejecución a T4 y vuelve a correr esta celda"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
%%writefile resolvedor_temporal.py
# -*- coding: utf-8 -*-
"""
resolvedor_temporal.py -- Herramienta determinista de resolucion temporal.

Vinculo con el diagnostico de Deliverable 1: el piso de error del modelo
(Qwen3-4B-Instruct-2507) esta concentrado en la resolucion de fechas
relativas; por eso el calculo temporal NO lo hace el modelo de lenguaje,
sino esta herramienta, con reglas auditables y reproducibles.

La regla de oro es: si la expresion no entra en la gramatica, se devuelve
None. Nunca se inventa un valor.

Referencia temporal del proyecto: 2026-09-01 (martes).
"""

from __future__ import annotations

import re
import unicodedata
from datetime import date, datetime, timedelta

FECHA_REF = "2026-09-01"
DIA_REF = "martes"

_MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10,
    "noviembre": 11, "diciembre": 12,
    "january": 1, "february": 2, "march": 3, "april": 4, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11,
    "december": 12,
}
_MESES_EN = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11,
    "december": 12,
}

# nombre normalizado -> isoweekday (1=lunes .. 7=domingo)
_DIAS = {
    "lunes": 1, "martes": 2, "miercoles": 3, "jueves": 4, "viernes": 5,
    "sabado": 6, "domingo": 7,
    "monday": 1, "tuesday": 2, "wednesday": 3, "thursday": 4, "friday": 5,
    "saturday": 6, "sunday": 7,
}

_NUMEROS = {
    "una": 1, "un": 1, "dos": 2, "tres": 3, "cuatro": 4, "cinco": 5,
    "seis": 6, "siete": 7, "ocho": 8, "nueve": 9, "diez": 10,
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
    "seven": 7, "eight": 8, "nine": 9, "ten": 10,
}

_MESES_ALT = "|".join(sorted(_MESES, key=len, reverse=True))
_MESES_EN_ALT = "|".join(sorted(_MESES_EN, key=len, reverse=True))
_DIAS_ALT = "|".join(sorted(_DIAS, key=len, reverse=True))
_NUMS_ALT = "|".join(sorted(_NUMEROS, key=len, reverse=True))

_RE_DDM = re.compile(r"\b(\d{1,2})[/-](\d{1,2})\b")
_RE_DIA_DE_MES = re.compile(r"\b(\d{1,2})\s+de\s+(" + _MESES_ALT + r")\b")
_RE_MES_DIA_EN = re.compile(r"\b(" + _MESES_EN_ALT + r")\s+(\d{1,2})(?:st|nd|rd|th)?\b")
_RE_DIA_DE_MES_EN = re.compile(r"\b(\d{1,2})(?:st|nd|rd|th)?\s+of\s+(" + _MESES_EN_ALT + r")\b")
_RE_PASADO_MANANA = re.compile(r"\bpasado\s+manana\b")
_RE_MANANA = re.compile(r"\bmanana\b")
_RE_EN_N = re.compile(r"\b(?:en|in)\s+(" + _NUMS_ALT + r"|\d+)\s+(dia|dias|day|days|semana|semanas|week|weeks)\b")
_RE_SEMANA_PROX = re.compile(r"\b(" + _DIAS_ALT + r")\s+(?:de\s+la\s+|of\s+the\s+)(?:proxima|next)\s+semana\b")
_RE_WEEKDAY = re.compile(r"\b(?:el\s+|la\s+|on\s+|this\s+|next\s+|este\s+|esta\s+|proximo\s+|proxima\s+)*(" + _DIAS_ALT + r")\b")
_RE_DIA_SOLO = re.compile(r"\bel\s+(?:dia\s+)?(\d{1,2})\b")

# Contextos que anulan una expresion (no son fechas del evento).
_EXCL_MANANA = ("de la ", "por la ", "en la ", "esta ", "la ", "the ", "this ")
_EXCL_WEEKDAY = ("los ", "las ", "cada ", "every ", "each ")

_RE_HORA_A_LAS = re.compile(
    r"\ba\s+las\s+(\d{1,2})(?::(\d{2}))?\s*"
    r"(am|pm|de\s+la\s+manana|de\s+la\s+tarde|de\s+la\s+noche)?"
)
_RE_HORA_AT = re.compile(r"\bat\s+(\d{1,2})(?::(\d{2}))?\s*(am|pm)?")
_RE_HORA_24H = re.compile(r"\b(\d{1,2}):(\d{2})\s*(am|pm)?")
_RE_HORA_MER = re.compile(r"\b(\d{1,2})\s*(am|pm)\b")
_RE_HORA_FRANJA = re.compile(r"\b(\d{1,2})\s*de\s+la\s+(manana|tarde|noche)\b")
_RE_RANGO = re.compile(r"\bde\s+(\d{1,2})\s+a\s+(\d{1,2})\b")


# ---------------------------------------------------------------------------
# utilidades
# ---------------------------------------------------------------------------

def _norm(texto: str) -> str:
    """Minusculas + sin tildes. Conserva la longitud (los spans siguen valiendo)."""
    s = texto.lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    return s


def _iso(d: date) -> str:
    return d.isoformat()


def _a_date(iso: str) -> date:
    return datetime.strptime(iso, "%Y-%m-%d").date()


def _con_anio(dia: int, mes: int, ref: date) -> str | None:
    """Fecha valida en el anio de ref; si ya paso, anio siguiente."""
    for anio in (ref.year, ref.year + 1):
        try:
            return _iso(date(anio, mes, dia))
        except ValueError:
            continue
    return None


def _proximo_weekday(ref: date, objetivo: int) -> date:
    """Siguiente ocurrencia estrictamente posterior a ref."""
    delta = (objetivo - ref.isoweekday()) % 7
    if delta == 0:
        delta = 7
    return ref + timedelta(days=delta)


def _weekday_semana_proxima(ref: date, objetivo: int) -> date:
    lunes_prox = ref - timedelta(days=ref.isoweekday() - 1) + timedelta(days=7)
    return lunes_prox + timedelta(days=objetivo - 1)


def _previo(t: str, inicio: int, contextos: tuple) -> bool:
    antes = t[max(0, inicio - 10):inicio]
    return any(antes.endswith(c) for c in contextos)


def _a_entero(valor: str) -> int | None:
    if valor.isdigit():
        return int(valor)
    return _NUMEROS.get(valor)


# ---------------------------------------------------------------------------
# busqueda de la expresion temporal de FECHA
# ---------------------------------------------------------------------------

def _buscar_fecha(t: str, ref: date) -> tuple[str | None, tuple[int, int] | None]:
    """Devuelve (fecha_iso, span) sobre el texto normalizado `t`."""

    # 1. formato dd/mm o dd-mm  (convencion del benchmark: dia/mes)
    for m in _RE_DDM.finditer(t):
        dia, mes = int(m.group(1)), int(m.group(2))
        if 1 <= dia <= 31 and 1 <= mes <= 12:
            f = _con_anio(dia, mes, ref)
            if f:
                return f, m.span()

    # 2. "15 de septiembre" / "september 8th" / "8th of september"
    for m in _RE_DIA_DE_MES.finditer(t):
        f = _con_anio(int(m.group(1)), _MESES[m.group(2)], ref)
        if f:
            return f, m.span()
    for m in _RE_MES_DIA_EN.finditer(t):
        f = _con_anio(int(m.group(2)), _MESES_EN[m.group(1)], ref)
        if f:
            return f, m.span()
    for m in _RE_DIA_DE_MES_EN.finditer(t):
        f = _con_anio(int(m.group(1)), _MESES_EN[m.group(2)], ref)
        if f:
            return f, m.span()

    # 3. "pasado manana"  (antes que "manana", si no daria +1)
    m = _RE_PASADO_MANANA.search(t)
    if m:
        return _iso(ref + timedelta(days=2)), m.span()

    # 4. "manana" (no "de la manana", "esta manana", ...)
    for m in _RE_MANANA.finditer(t):
        if _previo(t, m.start(), _EXCL_MANANA):
            continue
        return _iso(ref + timedelta(days=1)), m.span()

    # 5. "en dos semanas" / "in two weeks"
    m = _RE_EN_N.search(t)
    if m:
        n = _a_entero(m.group(1))
        if n:
            semanas = m.group(2).startswith(("semana", "week"))
            return _iso(ref + timedelta(days=7 * n if semanas else n)), m.span()

    # 6. "miercoles de la proxima semana"
    m = _RE_SEMANA_PROX.search(t)
    if m:
        return _iso(_weekday_semana_proxima(ref, _DIAS[m.group(1)])), m.span()

    # 7. "el lunes" / "proximo martes" / "on monday"
    for m in _RE_WEEKDAY.finditer(t):
        if _previo(t, m.start(), _EXCL_WEEKDAY):
            continue
        return _iso(_proximo_weekday(ref, _DIAS[m.group(1)])), m.span()

    # 8. "el 10" (solo dia; mes corriente si queda futuro)
    for m in _RE_DIA_SOLO.finditer(t):
        dia = int(m.group(1))
        if not 1 <= dia <= 31:
            continue
        mes, anio = ref.month, ref.year
        if dia < ref.day:
            mes += 1
            if mes == 13:
                mes, anio = 1, anio + 1
        try:
            return _iso(date(anio, mes, dia)), m.span()
        except ValueError:
            # dia invalido para el mes (p.ej. 31 de febrero) -> mes siguiente
            try:
                mes2 = mes + 1 if mes < 12 else 1
                anio2 = anio if mes < 12 else anio + 1
                return _iso(date(anio2, mes2, dia)), m.span()
            except ValueError:
                continue

    return None, None


# ---------------------------------------------------------------------------
# busqueda de la expresion temporal de HORA
# ---------------------------------------------------------------------------

def _aplicar_hora(h: int, m: int, sufijo: str | None) -> str | None:
    suf = (sufijo or "").replace("  ", " ").strip()
    suf = re.sub(r"\s+", " ", suf)
    if suf in ("pm", "de la tarde", "de la noche"):
        if h < 12:
            h += 12
    elif suf in ("am", "de la manana"):
        if h == 12:
            h = 0
    else:
        # convencion del benchmark: hora en blanco 1..7 = tarde
        if 1 <= h <= 7:
            h += 12
    if h > 23 or m > 59:
        return None
    return "%02d:%02d" % (h, m)


def _buscar_hora(t: str, spans: list[tuple[int, int]]) -> tuple[str | None, tuple[int, int] | None]:
    """Primera expresion horaria del texto, ignorando los spans de fecha."""
    candidatos = []

    for m in _RE_HORA_A_LAS.finditer(t):
        candidatos.append((m, m.group(1), m.group(2) or "0", m.group(3)))
    for m in _RE_HORA_AT.finditer(t):
        candidatos.append((m, m.group(1), m.group(2) or "0", m.group(3)))
    for m in _RE_HORA_24H.finditer(t):
        candidatos.append((m, m.group(1), m.group(2), m.group(3)))
    for m in _RE_HORA_MER.finditer(t):
        candidatos.append((m, m.group(1), "0", m.group(2)))
    for m in _RE_HORA_FRANJA.finditer(t):
        suf = "de la " + m.group(2)
        candidatos.append((m, m.group(1), "0", suf))
    for m in _RE_RANGO.finditer(t):
        candidatos.append((m, m.group(1), "0", None))

    def sin_solape(span):
        a, b = span
        return not any(a < fb and b > fa for fa, fb in spans)

    candidatos = [c for c in candidatos if sin_solape(c[0].span())]
    if not candidatos:
        return None, None

    m, hs, ms, sufijo = min(candidatos, key=lambda c: (c[0].start(), -c[0].end()))
    try:
        h, mm = int(hs), int(ms)
    except ValueError:
        return None, None
    hora = _aplicar_hora(h, mm, sufijo)
    return hora, m.span()


# ---------------------------------------------------------------------------
# API publica
# ---------------------------------------------------------------------------

def analizar_temporal(texto: str, fecha_ref: str = FECHA_REF) -> dict:
    """
    Resuelve la parte temporal de un mensaje.

    Devuelve:
        fecha        -> 'YYYY-MM-DD' o None
        hora_inicio  -> 'HH:MM' o None
        fecha_expr   -> expresion literal encontrada (para trazabilidad)
        hora_expr    -> expresion literal encontrada (para trazabilidad)
    """
    ref = _a_date(fecha_ref)
    t = _norm(texto)

    fecha, span_fecha = _buscar_fecha(t, ref)
    spans = [span_fecha] if span_fecha else []
    hora, span_hora = _buscar_hora(t, spans)

    def superficie(span):
        if not span:
            return None
        return texto[span[0]:span[1]].strip()

    return {
        "fecha": fecha,
        "hora_inicio": hora,
        "fecha_expr": superficie(span_fecha),
        "hora_expr": superficie(span_hora),
    }


def resolver_fecha(texto: str, fecha_ref: str = FECHA_REF) -> str | None:
    """Solo la fecha. None si la expresion no entra en la gramatica."""
    return analizar_temporal(texto, fecha_ref)["fecha"]


def resolver_hora_expr(expr: str) -> str | None:
    """Resuelve una expresion horaria aislada ('a las 4', '10am', '15:30')."""
    if not expr:
        return None
    hora, _ = _buscar_hora(_norm(expr), [])
    if hora:
        return hora
    return resolver_hora_valor(expr)


def resolver_hora_valor(valor) -> str | None:
    """Valor suelto que viene del modelo: '4', '09:30', '10 pm'."""
    if valor is None:
        return None
    if not isinstance(valor, str):
        return None
    m = re.fullmatch(r"\s*(\d{1,2})(?::(\d{2}))?\s*(am|pm)?\s*", valor, re.I)
    if not m:
        return None
    return _aplicar_hora(int(m.group(1)), int(m.group(2) or 0), m.group(3))


def resolver_hora(texto: str, fecha_ref: str = FECHA_REF) -> str | None:
    """Solo la hora, ignorando los numeros que forman parte de la fecha."""
    return analizar_temporal(texto, fecha_ref)["hora_inicio"]


if __name__ == "__main__":
    pruebas = [
        "Reunion el lunes a las 3pm",
        "agenda reunion manana 10am porfa",
        "Oye, agendame una junta con la Dani pasado manana a las 4 en la sala B-11",
        "Reunion el miércoles de la próxima semana a las 11",
        "Cita en dos semanas a las 12:00",
        "Llamada a las 14:30",
        "Junta de emergencia",
        "Call with the client on Monday at 5pm",
        "Reunión el 10/09 a las 10 pm",
        "Reunión el lunes 7 de septiembre sin hora definida",
    ]
    for p in pruebas:
        print("%-70s -> %s" % (p, analizar_temporal(p)))

In [ ]:
%%writefile evaluar.py
"""
evaluar.py - Script de evaluación del benchmark de agendamiento de reuniones.

Uso:
  1. Generar salidas del modelo: python evaluar.py --generar
  2. Evaluar resultados:        python evaluar.py --evaluar
  3. Comparar modelos:          python evaluar.py --comparar

Archivos esperados:
  benchmark/frases.json          -> Benchmark con ground truth
  benchmark/resultados_{modelo}.json -> Salidas del modelo a evaluar
"""

import json
import os
import sys
from datetime import datetime, timedelta
from collections import defaultdict

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
BENCHMARK_PATH = os.path.join(SCRIPT_DIR, "frases.json")
RESULTADOS_DIR = SCRIPT_DIR

CAMPOS_OBLIGATORIOS = [
    "titulo", "fecha", "hora_inicio", "duracion_minutos",
    "participantes", "ubicacion", "recordatorio_minutos", "zona_horaria"
]


def cargar_benchmark():
    with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["frases"], data["metadata"]


def cargar_resultados(nombre_modelo):
    path = os.path.join(RESULTADOS_DIR, f"resultados_{nombre_modelo}.json")
    if not os.path.exists(path):
        print(f"[ERROR] No se encontro: {path}")
        print(f"        Primero genera las salidas con: python evaluar.py --generar")
        sys.exit(1)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def comparar_campos(esperado, obtenido):
    """Compara dos valores de campo y retorna (correcto, tipo_error)."""
    if esperado is None and obtenido is None:
        return True, None
    if esperado is None and obtenido is not None:
        return False, "inventado"
    if esperado is not None and obtenido is None:
        return False, "omision"

    if isinstance(esperado, list):
        if not isinstance(obtenido, list):
            return False, "formato_incorrecto"
        esp_set = set(str(x).strip().lower() for x in esperado)
        opt_set = set(str(x).strip().lower() for x in obtenido)
        if esp_set == opt_set:
            return True, None
        faltantes = esp_set - opt_set
        extra = opt_set - esp_set
        if faltantes:
            return False, "omision"
        if extra:
            return False, "inventado"
        return False, "formato_incorrecto"

    esp_str = str(esperado).strip().lower()
    opt_str = str(obtenido).strip().lower()

    if esp_str == opt_str:
        return True, None

    if isinstance(esperado, str) and isinstance(obtenido, str):
        if esp_str in opt_str or opt_str in esp_str:
            return True, None

    return False, "valor_incorrecto"


def evaluar_resultado(frase, resultado_modelo):
    """Evalúa un resultado contra el ground truth. Retorna dict con detalles."""
    gt = frase["ground_truth"]
    detalles = []
    campos_correctos = 0
    total_campos = len(CAMPOS_OBLIGATORIOS)

    for campo in CAMPOS_OBLIGATORIOS:
        esperado = gt.get(campo)
        obtenido = resultado_modelo.get(campo)

        # Caso especial: duracion_minutos puede inferirse de hora_inicio + duracion
        if campo == "duracion_minutos" and obtenido is None and esperado is not None:
            # No penalizar si el modelo no infiere duración
            pass

        correcto, tipo_error = comparar_campos(esperado, obtenido)
        if correcto:
            campos_correctos += 1

        detalles.append({
            "campo": campo,
            "esperado": esperado,
            "obtenido": obtenido,
            "correcto": correcto,
            "tipo_error": tipo_error
        })

    es_correcto_total = campos_correctos == total_campos

    return {
        "id": frase["id"],
        "texto": frase["texto"],
        "categoria": frase["categoria"],
        "campos_correctos": campos_correctos,
        "total_campos": total_campos,
        "es_correcto_total": es_correcto_total,
        "detalles": detalles
    }


def calcular_metricas(resultados):
    """Calcula métricas agregadas de todos los resultados."""
    total = len(resultados)
    correctos_total = sum(1 for r in resultados if r["es_correcto_total"])

    errores_por_tipo = defaultdict(int)
    errores_por_categoria = defaultdict(lambda: {"total": 0, "correctos": 0})
    campos_acertados = defaultdict(int)
    campos_total = defaultdict(int)

    for r in resultados:
        cat = r["categoria"]
        errores_por_categoria[cat]["total"] += 1
        if r["es_correcto_total"]:
            errores_por_categoria[cat]["correctos"] += 1

        for d in r["detalles"]:
            campo = d["campo"]
            campos_total[campo] += 1
            if d["correcto"]:
                campos_acertados[campo] += 1
            elif d["tipo_error"]:
                errores_por_tipo[d["tipo_error"]] += 1

    metricas = {
        "total_frases": total,
        "correctos_totales": correctos_total,
        "precision_total": round(correctos_total / total * 100, 1) if total > 0 else 0,
        "precision_por_categoria": {},
        "precision_por_campo": {},
        "errores_por_tipo": dict(errores_por_tipo)
    }

    for cat, vals in errores_por_categoria.items():
        metricas["precision_por_categoria"][cat] = {
            "total": vals["total"],
            "correctos": vals["correctos"],
            "precision": round(vals["correctos"] / vals["total"] * 100, 1) if vals["total"] > 0 else 0
        }

    for campo in CAMPOS_OBLIGATORIOS:
        t = campos_total.get(campo, 0)
        c = campos_acertados.get(campo, 0)
        metricas["precision_por_campo"][campo] = {
            "total": t,
            "correctos": c,
            "precision": round(c / t * 100, 1) if t > 0 else 0
        }

    return metricas


def imprimir_reporte(metricas, nombre_modelo="modelo"):
    """Imprime el reporte en consola."""
    print("\n" + "=" * 60)
    print(f"  REPORTE DE EVALUACION: {nombre_modelo.upper()}")
    print("=" * 60)

    print(f"\n  Total frases:    {metricas['total_frases']}")
    print(f"  Correctas:       {metricas['correctos_totales']}")
    print(f"  Precision total: {metricas['precision_total']}%")

    print("\n  --- Precision por categoria ---")
    for cat, vals in sorted(metricas["precision_por_categoria"].items()):
        barra = "#" * int(vals["precision"] / 5)
        print(f"  {cat:20s}: {vals['precision']:5.1f}% ({vals['correctos']}/{vals['total']}) {barra}")

    print("\n  --- Precision por campo ---")
    for campo, vals in metricas["precision_por_campo"].items():
        barra = "#" * int(vals["precision"] / 5)
        print(f"  {campo:25s}: {vals['precision']:5.1f}% ({vals['correctos']}/{vals['total']}) {barra}")

    if metricas["errores_por_tipo"]:
        print("\n  --- Errores por tipo ---")
        for tipo, cantidad in sorted(metricas["errores_por_tipo"].items(), key=lambda x: -x[1]):
            print(f"  {tipo:25s}: {cantidad}")

    print("\n" + "=" * 60)


def guardar_reporte(metricas, nombre_modelo, resultados_detalles):
    """Guarda el reporte como JSON."""
    reporte = {
        "modelo": nombre_modelo,
        "fecha_evaluacion": datetime.now().isoformat(),
        "metricas": metricas,
        "detalles_por_frase": resultados_detalles
    }
    path = os.path.join(RESULTADOS_DIR, f"reporte_{nombre_modelo}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(reporte, f, indent=2, ensure_ascii=False)
    print(f"\n  Reporte guardado en: {path}")


def modo_generar():
    """Genera una plantilla de resultados para que el usuario la complete."""
    frases, _ = cargar_benchmark()
    plantilla = []
    for frase in frases:
        plantilla.append({
            "id": frase["id"],
            "texto": frase["texto"],
            "resultado_modelo": {
                "titulo": None,
                "fecha": None,
                "hora_inicio": None,
                "duracion_minutos": None,
                "participantes": [],
                "ubicacion": None,
                "recordatorio_minutos": None,
                "zona_horaria": None
            }
        })

    path = os.path.join(RESULTADOS_DIR, "resultados_PLANTILLA.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(plantilla, f, indent=2, ensure_ascii=False)
    print(f"Plantilla generada: {path}")
    print("Completa 'resultado_modelo' para cada frase y renombra a resultados_{nombre_modelo}.json")


def modo_evaluar(nombre_modelo):
    """Evalúa los resultados de un modelo específico."""
    frases, metadata = cargar_benchmark()
    resultados_raw = cargar_resultados(nombre_modelo)

    mapa_resultados = {}
    for r in resultados_raw:
        mapa_resultados[r["id"]] = r["resultado_modelo"]

    resultados_evaluados = []
    for frase in frases:
        rid = frase["id"]
        if rid in mapa_resultados:
            ev = evaluar_resultado(frase, mapa_resultados[rid])
            resultados_evaluados.append(ev)
        else:
            print(f"[WARN] Frase {rid} sin resultado del modelo")

    metricas = calcular_metricas(resultados_evaluados)
    imprimir_reporte(metricas, nombre_modelo)
    guardar_reporte(metricas, nombre_modelo, resultados_evaluados)

    return metricas


def modo_comparar(modelos):
    """Compara múltiples modelos lado a lado."""
    todos_metricas = {}
    for modelo in modelos:
        frases, _ = cargar_benchmark()
        resultados_raw = cargar_resultados(modelo)
        mapa = {r["id"]: r["resultado_modelo"] for r in resultados_raw}

        evaluados = []
        for frase in frases:
            if frase["id"] in mapa:
                evaluados.append(evaluar_resultado(frase, mapa[frase["id"]]))

        todos_metricas[modelo] = calcular_metricas(evaluados)

    print("\n" + "=" * 70)
    print("  COMPARACION DE MODELOS")
    print("=" * 70)
    print(f"\n  {'Modelo':20s} {'Precision':>10s} {'Correctas':>10s} {'Total':>8s}")
    print("  " + "-" * 50)
    for modelo, m in sorted(todos_metricas.items(), key=lambda x: -x[1]["precision_total"]):
        print(f"  {modelo:20s} {m['precision_total']:>9.1f}% {m['correctos_totales']:>9d} {m['total_frases']:>7d}")

    print("\n  --- Precision por campo ---")
    campos = CAMPOS_OBLIGATORIOS
    header = f"  {'Campo':25s}"
    for modelo in modelos:
        header += f" {modelo[:10]:>12s}"
    print(header)
    print("  " + "-" * (25 + 13 * len(modelos)))

    for campo in campos:
        linea = f"  {campo:25s}"
        for modelo in modelos:
            p = todos_metricas[modelo]["precision_por_campo"].get(campo, {}).get("precision", 0)
            linea += f" {p:>11.1f}%"
        print(linea)

    print("\n" + "=" * 70)


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("\nUso:")
        print("  python evaluar.py --generar                    # Genera plantilla")
        print("  python evaluar.py --evaluar <nombre_modelo>    # Evalúa un modelo")
        print("  python evaluar.py --comparar m1 m2 m3          # Compara modelos")
        print("\nEjemplo:")
        print("  python evaluar.py --evaluar tinyllama")
        sys.exit(0)

    if sys.argv[1] == "--generar":
        modo_generar()
    elif sys.argv[1] == "--evaluar" and len(sys.argv) >= 3:
        modo_evaluar(sys.argv[2])
    elif sys.argv[1] == "--comparar" and len(sys.argv) >= 4:
        modo_comparar(sys.argv[2:])
    else:
        print("[ERROR] Argumentos invalidos. Ejecuta sin argumentos para ver ayuda.")

In [ ]:
%%writefile benchmark_ft.py
"""
benchmark_ft.py - Pruebas automaticas del antes/despues del fine-tuning (LoRA).

Cada frase de un benchmark es una prueba: PASA si los 8 campos coinciden con el ground truth
segun evaluar.py del D1 (mismo criterio para antes y despues). Lo usan benchmark_antes.ipynb y
benchmark_despues.ipynb en Colab; en local sirve para volver a revisar los resultados bajados.

Uso local:
  python benchmark_ft.py --probar   <benchmark.json> <resultados.json>
  python benchmark_ft.py --comparar <benchmark.json> <resultados_antes.json> <resultados_despues.json>
  python benchmark_ft.py --techo    <benchmark.json>     # fechas que acierta el resolvedor (no depende del modelo)

Formato de resultados (compatible con evaluar.py): [{id, texto, resultado_modelo, crudo, segundos, json_valido}]
"""

import json
import os
import sys
from math import comb

import evaluar as ev
import resolvedor_temporal as rt


def cargar(ruta):
    with open(ruta, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["frases"], data["metadata"]


def guardar(obj, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def correr(ruta_benchmark, extraer):
    """extraer(texto, fecha_ref) -> (evento, info) con info = {crudo, segundos, error}."""
    frases, meta = cargar(ruta_benchmark)
    resultados = []
    for i, frase in enumerate(frases, 1):
        evento, info = extraer(frase["texto"], meta["fecha_referencia"])
        resultados.append({
            "id": frase["id"], "texto": frase["texto"], "resultado_modelo": evento,
            "crudo": info["crudo"], "segundos": round(info["segundos"], 2), "json_valido": not info["error"],
        })
        print(f"  [{i:2d}/{len(frases)}] {info['segundos']:5.1f} s  {frase['texto'][:70]}")
    return resultados


def _evaluar(frases, resultados):
    mapa = {r["id"]: r["resultado_modelo"] for r in resultados}
    return [ev.evaluar_resultado(f, mapa.get(f["id"]) or {}) for f in frases]


def techo_fecha(frases, fecha_ref):
    """Fechas correctas del resolvedor solo: igual antes y despues (el modelo no calcula la fecha)."""
    return sum(rt.resolver_fecha(f["texto"], fecha_ref) == f["ground_truth"]["fecha"] for f in frases)


def probar(ruta_benchmark, resultados, nombre, detalle=True):
    """Corre las pruebas: PASA/FALLA por frase y metricas de evaluar.py. Devuelve (metricas, evaluados)."""
    frases, meta = cargar(ruta_benchmark)
    evaluados = _evaluar(frases, resultados)
    if detalle:
        print(f"\nPRUEBAS: {nombre} sobre {os.path.basename(ruta_benchmark)} ({len(frases)} frases)")
        for e in evaluados:
            print(f"  {'PASA ' if e['es_correcto_total'] else 'FALLA'} #{e['id']:<3d} {e['texto'][:80]}")
            for d in e["detalles"]:
                if not d["correcto"]:
                    print(f"          {d['campo']}: esperado {d['esperado']!r} -> obtenido {d['obtenido']!r} ({d['tipo_error']})")
    metricas = ev.calcular_metricas(evaluados)
    metricas["json_valido"] = sum(1 for r in resultados if r.get("json_valido", True))
    tiempos = [r["segundos"] for r in resultados if "segundos" in r]
    metricas["segundos_promedio"] = round(sum(tiempos) / len(tiempos), 2) if tiempos else None
    metricas["techo_fecha_resolvedor"] = techo_fecha(frases, meta["fecha_referencia"])
    ev.imprimir_reporte(metricas, nombre)
    print(f"  JSON valido: {metricas['json_valido']}/{len(frases)} | segundos por frase: {metricas['segundos_promedio']}"
          f" | fechas que acierta el resolvedor solo: {metricas['techo_fecha_resolvedor']}/{len(frases)}")
    return metricas, evaluados


def _mcnemar(b, c):
    """p-valor exacto (binomial, dos colas) sobre los pares discordantes."""
    n = b + c
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(comb(n, k) for k in range(min(b, c) + 1)) / 2 ** n)


def comparar(ruta_benchmark, res_antes, res_despues):
    """Antes vs despues sobre las mismas frases y el mismo criterio. Devuelve un resumen (dict)."""
    frases, meta = cargar(ruta_benchmark)
    ea, ed = _evaluar(frases, res_antes), _evaluar(frases, res_despues)
    ma, md = ev.calcular_metricas(ea), ev.calcular_metricas(ed)
    n = len(frases)

    def pct(x):
        return f"{x:5.1f}%"

    print("\n" + "=" * 72)
    print(f"  ANTES vs DESPUES DEL FINE-TUNING - {os.path.basename(ruta_benchmark)} ({n} frases)")
    print("=" * 72)
    print(f"  {'':28s} {'antes':>10s} {'despues':>10s} {'cambio':>10s}")
    print(f"  {'frases 100% correctas':28s} {ma['correctos_totales']:>6d}/{n:<3d} {md['correctos_totales']:>6d}/{n:<3d}"
          f" {md['correctos_totales'] - ma['correctos_totales']:>+10d}")
    print(f"  {'exactitud':28s} {pct(ma['precision_total']):>10s} {pct(md['precision_total']):>10s}"
          f" {md['precision_total'] - ma['precision_total']:>+9.1f}pp")
    ja = sum(1 for r in res_antes if r.get("json_valido", True))
    jd = sum(1 for r in res_despues if r.get("json_valido", True))
    print(f"  {'JSON valido':28s} {ja:>6d}/{n:<3d} {jd:>6d}/{n:<3d}")
    ta = [r["segundos"] for r in res_antes if "segundos" in r]
    td = [r["segundos"] for r in res_despues if "segundos" in r]
    if ta and td:
        print(f"  {'segundos por frase':28s} {sum(ta) / len(ta):>10.2f} {sum(td) / len(td):>10.2f}")

    print("\n  --- Exactitud por campo ---")
    for campo in ev.CAMPOS_OBLIGATORIOS:
        a, d = ma["precision_por_campo"][campo]["precision"], md["precision_por_campo"][campo]["precision"]
        nota = "  (resolvedor: igual en ambos)" if campo == "fecha" else "  (reglas: igual en ambos)" if campo == "zona_horaria" else ""
        print(f"  {campo:28s} {pct(a):>10s} {pct(d):>10s} {d - a:>+9.1f}pp{nota}")

    print("\n  --- Exactitud por categoria ---")
    for cat in sorted(md["precision_por_categoria"]):
        a, d = ma["precision_por_categoria"][cat], md["precision_por_categoria"][cat]
        print(f"  {cat:28s} {a['correctos']:>6d}/{a['total']:<3d} {d['correctos']:>6d}/{d['total']:<3d}"
              f" {d['correctos'] - a['correctos']:>+10d}")

    arreglo = [(x, y) for x, y in zip(ea, ed) if not x["es_correcto_total"] and y["es_correcto_total"]]
    rompio = [(x, y) for x, y in zip(ea, ed) if x["es_correcto_total"] and not y["es_correcto_total"]]
    p = _mcnemar(len(rompio), len(arreglo))
    print(f"\n  Frases que el fine-tuning ARREGLO ({len(arreglo)}):")
    for x, _ in arreglo:
        fallos = ", ".join(d["campo"] for d in x["detalles"] if not d["correcto"])
        print(f"    #{x['id']:<3d} {x['texto'][:70]}  [antes fallaba: {fallos}]")
    print(f"  Frases que el fine-tuning ROMPIO ({len(rompio)}):")
    for _, y in rompio:
        fallos = ", ".join(d["campo"] for d in y["detalles"] if not d["correcto"])
        print(f"    #{y['id']:<3d} {y['texto'][:70]}  [ahora falla: {fallos}]")
    print(f"\n  McNemar exacto (pares discordantes {len(arreglo)} vs {len(rompio)}): p = {p:.3f}"
          + ("  -> diferencia significativa (p < 0.05)" if p < 0.05 else "  -> no alcanza significancia (p >= 0.05)"))
    print("=" * 72)

    return {
        "benchmark": os.path.basename(ruta_benchmark), "total_frases": n,
        "antes": ma, "despues": md,
        "arreglo": [x["id"] for x, _ in arreglo], "rompio": [y["id"] for _, y in rompio],
        "mcnemar_p": p,
    }


if __name__ == "__main__":
    sys.stdout.reconfigure(encoding="utf-8")
    args = sys.argv[1:]
    if len(args) == 3 and args[0] == "--probar":
        with open(args[2], encoding="utf-8") as f:
            probar(args[1], json.load(f), os.path.splitext(os.path.basename(args[2]))[0])
    elif len(args) == 4 and args[0] == "--comparar":
        with open(args[2], encoding="utf-8") as f, open(args[3], encoding="utf-8") as g:
            comparar(args[1], json.load(f), json.load(g))
    elif len(args) == 2 and args[0] == "--techo":
        frases, meta = cargar(args[1])
        print(f"fechas correctas del resolvedor: {techo_fecha(frases, meta['fecha_referencia'])}/{len(frases)}")
    else:
        print(__doc__)
        sys.exit(1)

In [ ]:
%%writefile frases.json
{
  "metadata": {
    "descripcion": "Benchmark de frases para evaluar parsing de eventos de calendario desde lenguaje natural (set Benchmark2, incluye mensajes estilo chat WhatsApp/Telegram)",
    "fecha_referencia": "2026-09-01",
    "dia_referencia": "martes",
    "total_frases": 45,
    "categorias": {
      "simple": "Fechas absolutas, sin participantes, campos basicos",
      "con_participantes": "Incluye personas explicitas",
      "ambigua": "Expresiones temporales relativas (manana, pasado manana, proxima semana, etc.)",
      "compleja": "Multiples campos, detalles especificos",
      "edge_case": "Casos borde: info faltante, formato raro, idioma ingles, zona horaria distinta",
      "chat": "Mensajes informales estilo WhatsApp/Telegram"
    }
  },
  "frases": [
    {
      "id": 1,
      "categoria": "simple",
      "texto": "Reunión el 4 de septiembre a las 9 am",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-09-04",
        "hora_inicio": "09:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 2,
      "categoria": "simple",
      "texto": "Cita médica el 15 de septiembre a las 10:30",
      "ground_truth": {
        "titulo": "Cita médica",
        "fecha": "2026-09-15",
        "hora_inicio": "10:30",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 3,
      "categoria": "simple",
      "texto": "Junta el 7 de septiembre a las 16:00",
      "ground_truth": {
        "titulo": "Junta",
        "fecha": "2026-09-07",
        "hora_inicio": "16:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 4,
      "categoria": "simple",
      "texto": "Meeting on September 8th at 2pm",
      "ground_truth": {
        "titulo": "Meeting",
        "fecha": "2026-09-08",
        "hora_inicio": "14:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 5,
      "categoria": "simple",
      "texto": "Entrevista el 10/09 a las 11:15",
      "ground_truth": {
        "titulo": "Entrevista",
        "fecha": "2026-09-10",
        "hora_inicio": "11:15",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 6,
      "categoria": "simple",
      "texto": "Taller el sábado 12 de septiembre a las 3 de la tarde",
      "ground_truth": {
        "titulo": "Taller",
        "fecha": "2026-09-12",
        "hora_inicio": "15:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 7,
      "categoria": "simple",
      "texto": "Presentación el 3 de septiembre a las 8:00",
      "ground_truth": {
        "titulo": "Presentación",
        "fecha": "2026-09-03",
        "hora_inicio": "08:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 8,
      "categoria": "simple",
      "texto": "Sesión de fotos el 11 de septiembre a las 17:30",
      "ground_truth": {
        "titulo": "Sesión de fotos",
        "fecha": "2026-09-11",
        "hora_inicio": "17:30",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 9,
      "categoria": "con_participantes",
      "texto": "Reunión con Ana y Pedro el 2 de septiembre a las 10",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-09-02",
        "hora_inicio": "10:00",
        "duracion_minutos": null,
        "participantes": ["Ana", "Pedro"],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 10,
      "categoria": "con_participantes",
      "texto": "Junta con el equipo de ventas el jueves 10 de septiembre a las 15:30",
      "ground_truth": {
        "titulo": "Junta",
        "fecha": "2026-09-10",
        "hora_inicio": "15:30",
        "duracion_minutos": null,
        "participantes": ["equipo de ventas"],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 11,
      "categoria": "con_participantes",
      "texto": "Llamada con el cliente el 4 de septiembre a las 11",
      "ground_truth": {
        "titulo": "Llamada",
        "fecha": "2026-09-04",
        "hora_inicio": "11:00",
        "duracion_minutos": null,
        "participantes": ["cliente"],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 12,
      "categoria": "con_participantes",
      "texto": "Almuerzo con María el viernes a las 13:00",
      "ground_truth": {
        "titulo": "Almuerzo",
        "fecha": "2026-09-04",
        "hora_inicio": "13:00",
        "duracion_minutos": null,
        "participantes": ["María"],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 13,
      "categoria": "con_participantes",
      "texto": "Reunión con Dani y el profe el 8 de septiembre a las 9:30",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-09-08",
        "hora_inicio": "09:30",
        "duracion_minutos": null,
        "participantes": ["Dani", "el profe"],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 14,
      "categoria": "con_participantes",
      "texto": "Café con Juan y Carla el lunes a las 17:00",
      "ground_truth": {
        "titulo": "Café",
        "fecha": "2026-09-07",
        "hora_inicio": "17:00",
        "duracion_minutos": null,
        "participantes": ["Juan", "Carla"],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 15,
      "categoria": "con_participantes",
      "texto": "Review con los desarrolladores el 10 de septiembre a las 14:00",
      "ground_truth": {
        "titulo": "Review",
        "fecha": "2026-09-10",
        "hora_inicio": "14:00",
        "duracion_minutos": null,
        "participantes": ["los desarrolladores"],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 16,
      "categoria": "ambigua",
      "texto": "Reunión mañana a las 10",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-09-02",
        "hora_inicio": "10:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 17,
      "categoria": "ambigua",
      "texto": "Junta el lunes a las 9",
      "ground_truth": {
        "titulo": "Junta",
        "fecha": "2026-09-07",
        "hora_inicio": "09:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 18,
      "categoria": "ambigua",
      "texto": "Entrevista pasado mañana a las 15:30",
      "ground_truth": {
        "titulo": "Entrevista",
        "fecha": "2026-09-03",
        "hora_inicio": "15:30",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 19,
      "categoria": "ambigua",
      "texto": "Clase el viernes a las 8:30",
      "ground_truth": {
        "titulo": "Clase",
        "fecha": "2026-09-04",
        "hora_inicio": "08:30",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 20,
      "categoria": "ambigua",
      "texto": "Reunión el miércoles de la próxima semana a las 11",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-09-09",
        "hora_inicio": "11:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 21,
      "categoria": "ambigua",
      "texto": "Cita en dos semanas a las 12:00",
      "ground_truth": {
        "titulo": "Cita",
        "fecha": "2026-09-15",
        "hora_inicio": "12:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 22,
      "categoria": "ambigua",
      "texto": "Dentista el próximo martes a las 16:30",
      "ground_truth": {
        "titulo": "Dentista",
        "fecha": "2026-09-08",
        "hora_inicio": "16:30",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 23,
      "categoria": "compleja",
      "texto": "Reunión de planificación el 15 de septiembre a las 11:30 por 1 hora con Ana y Pedro en la sala B-11",
      "ground_truth": {
        "titulo": "Reunión de planificación",
        "fecha": "2026-09-15",
        "hora_inicio": "11:30",
        "duracion_minutos": 60,
        "participantes": ["Ana", "Pedro"],
        "ubicacion": "sala B-11",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 24,
      "categoria": "compleja",
      "texto": "Presentación de tesis el jueves 10 de septiembre a las 10:00, 2 horas, con el comité evaluador, en el auditorio, avísame 30 minutos antes",
      "ground_truth": {
        "titulo": "Presentación de tesis",
        "fecha": "2026-09-10",
        "hora_inicio": "10:00",
        "duracion_minutos": 120,
        "participantes": ["comité evaluador"],
        "ubicacion": "auditorio",
        "recordatorio_minutos": 30,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 25,
      "categoria": "compleja",
      "texto": "Junta de sincronización el 8 de septiembre a las 9 con todo el equipo por videollamada, 45 minutos",
      "ground_truth": {
        "titulo": "Junta de sincronización",
        "fecha": "2026-09-08",
        "hora_inicio": "09:00",
        "duracion_minutos": 45,
        "participantes": ["todo el equipo"],
        "ubicacion": "videollamada",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 26,
      "categoria": "compleja",
      "texto": "Almuerzo de trabajo con los clientes el viernes 4 de septiembre a las 13:30 en el restaurante La Tabla por 1 hora y media",
      "ground_truth": {
        "titulo": "Almuerzo de trabajo",
        "fecha": "2026-09-04",
        "hora_inicio": "13:30",
        "duracion_minutos": 90,
        "participantes": ["los clientes"],
        "ubicacion": "restaurante La Tabla",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 27,
      "categoria": "compleja",
      "texto": "Workshop de IA el lunes 14 de septiembre de 9 a 12 en el salón 201",
      "ground_truth": {
        "titulo": "Workshop de IA",
        "fecha": "2026-09-14",
        "hora_inicio": "09:00",
        "duracion_minutos": 180,
        "participantes": [],
        "ubicacion": "salón 201",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 28,
      "categoria": "compleja",
      "texto": "Entrevista técnica el 11 de septiembre a las 16:00 por Zoom con el equipo de RRHH, 1 hora, recordatorio 10 minutos antes",
      "ground_truth": {
        "titulo": "Entrevista técnica",
        "fecha": "2026-09-11",
        "hora_inicio": "16:00",
        "duracion_minutos": 60,
        "participantes": ["equipo de RRHH"],
        "ubicacion": "Zoom",
        "recordatorio_minutos": 10,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 29,
      "categoria": "compleja",
      "texto": "Reunión de directorio el martes 8 de septiembre a las 18:30 en la oficina central, 2 horas, con los 5 directores",
      "ground_truth": {
        "titulo": "Reunión de directorio",
        "fecha": "2026-09-08",
        "hora_inicio": "18:30",
        "duracion_minutos": 120,
        "participantes": ["los 5 directores"],
        "ubicacion": "oficina central",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 30,
      "categoria": "edge_case",
      "texto": "Reunión el 15 de septiembre",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-09-15",
        "hora_inicio": null,
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 31,
      "categoria": "edge_case",
      "texto": "Llamada a las 14:30",
      "ground_truth": {
        "titulo": "Llamada",
        "fecha": null,
        "hora_inicio": "14:30",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 32,
      "categoria": "edge_case",
      "texto": "Junta de emergencia",
      "ground_truth": {
        "titulo": "Junta de emergencia",
        "fecha": null,
        "hora_inicio": null,
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 33,
      "categoria": "edge_case",
      "texto": "Call with the client on Monday at 5pm",
      "ground_truth": {
        "titulo": "Call",
        "fecha": "2026-09-07",
        "hora_inicio": "17:00",
        "duracion_minutos": null,
        "participantes": ["client"],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 34,
      "categoria": "edge_case",
      "texto": "Reunión el 5 de septiembre a las 9 en horario de Madrid",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-09-05",
        "hora_inicio": "09:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "Europe/Madrid"
      }
    },
    {
      "id": 35,
      "categoria": "edge_case",
      "texto": "Reunión el 10/09 a las 10 pm",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-09-10",
        "hora_inicio": "22:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 36,
      "categoria": "edge_case",
      "texto": "Reunión el lunes 7 de septiembre sin hora definida",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-09-07",
        "hora_inicio": null,
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 37,
      "categoria": "edge_case",
      "texto": "Junta virtual mañana",
      "ground_truth": {
        "titulo": "Junta",
        "fecha": "2026-09-02",
        "hora_inicio": null,
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": "virtual",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 38,
      "categoria": "chat",
      "texto": "Oye, agéndame una junta con la Dani pasado mañana a las 4 en la sala B-11",
      "ground_truth": {
        "titulo": "junta",
        "fecha": "2026-09-03",
        "hora_inicio": "16:00",
        "duracion_minutos": null,
        "participantes": ["Dani"],
        "ubicacion": "sala B-11",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 39,
      "categoria": "chat",
      "texto": "agenda reunion mañana 10am porfa",
      "ground_truth": {
        "titulo": "reunion",
        "fecha": "2026-09-02",
        "hora_inicio": "10:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 40,
      "categoria": "chat",
      "texto": "q onda, quedamos el viernes a las 7 en lo de siempre?",
      "ground_truth": {
        "titulo": "Quedada",
        "fecha": "2026-09-04",
        "hora_inicio": "19:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": "lo de siempre",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 41,
      "categoria": "chat",
      "texto": "hey schedule a call with the team on Friday 3pm",
      "ground_truth": {
        "titulo": "call",
        "fecha": "2026-09-04",
        "hora_inicio": "15:00",
        "duracion_minutos": null,
        "participantes": ["team"],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 42,
      "categoria": "chat",
      "texto": "acuerdate que el lunes tenemos la presentación a las 9",
      "ground_truth": {
        "titulo": "presentación",
        "fecha": "2026-09-07",
        "hora_inicio": "09:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 43,
      "categoria": "chat",
      "texto": "puedes agendar la reunión de sincronización el 10 a las 14? avísame 15 min antes",
      "ground_truth": {
        "titulo": "reunión de sincronización",
        "fecha": "2026-09-10",
        "hora_inicio": "14:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": 15,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 44,
      "categoria": "chat",
      "texto": "junta virtual mañana a las 5 con el profe",
      "ground_truth": {
        "titulo": "junta",
        "fecha": "2026-09-02",
        "hora_inicio": "17:00",
        "duracion_minutos": null,
        "participantes": ["el profe"],
        "ubicacion": "virtual",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 45,
      "categoria": "chat",
      "texto": "reunión con Dani el jueves a las 11, 1 hora",
      "ground_truth": {
        "titulo": "reunión",
        "fecha": "2026-09-03",
        "hora_inicio": "11:00",
        "duracion_minutos": 60,
        "participantes": ["Dani"],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    }
  ]
}

In [ ]:
%%writefile benchmark_ft.json
{
  "metadata": {
    "descripcion": "Benchmark de generalizacion para comparar Qwen3-4B antes y despues del fine-tuning (LoRA qwen3-4b-semantic-lora). Frases nuevas: ninguna aparece en train/dev/hard_dev del LoRA ni en frases.json, y la mayoria de titulos, participantes y ubicaciones no aparecen como objetivo en el entrenamiento. Mismo contrato de 8 campos y mismo criterio que frases.json (evaluar.py del D1).",
    "fecha_referencia": "2026-09-01",
    "dia_referencia": "martes",
    "total_frases": 48,
    "categorias": {
      "simple": "Fechas absolutas, sin participantes, campos basicos",
      "con_participantes": "Incluye personas explicitas",
      "ambigua": "Expresiones temporales relativas",
      "compleja": "Multiples campos, detalles especificos",
      "edge_case": "Info faltante, fecha u hora invalida, ingles, zona horaria distinta, hora AM temprana",
      "chat": "Mensajes informales estilo WhatsApp/Telegram"
    }
  },
  "frases": [
    {
      "id": 1,
      "categoria": "simple",
      "texto": "Control dental el 17 de septiembre a las 11:45",
      "ground_truth": {
        "titulo": "Control dental",
        "fecha": "2026-09-17",
        "hora_inicio": "11:45",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 2,
      "categoria": "simple",
      "texto": "Ensayo de la banda el 19/09 a las 18:00",
      "ground_truth": {
        "titulo": "Ensayo de la banda",
        "fecha": "2026-09-19",
        "hora_inicio": "18:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 3,
      "categoria": "simple",
      "texto": "Tutoría el 22 de septiembre a las 12:15",
      "ground_truth": {
        "titulo": "Tutoría",
        "fecha": "2026-09-22",
        "hora_inicio": "12:15",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 4,
      "categoria": "simple",
      "texto": "Webinar on September 23rd at 4pm",
      "ground_truth": {
        "titulo": "Webinar",
        "fecha": "2026-09-23",
        "hora_inicio": "16:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 5,
      "categoria": "simple",
      "texto": "Examen de grado el 29/09 a las 9:30",
      "ground_truth": {
        "titulo": "Examen de grado",
        "fecha": "2026-09-29",
        "hora_inicio": "09:30",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 6,
      "categoria": "simple",
      "texto": "Kickoff del proyecto el 2 de octubre a las 10 am",
      "ground_truth": {
        "titulo": "Kickoff",
        "fecha": "2026-10-02",
        "hora_inicio": "10:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 7,
      "categoria": "simple",
      "texto": "Vacuna del perro el 25 de septiembre a las 17:15",
      "ground_truth": {
        "titulo": "Vacuna del perro",
        "fecha": "2026-09-25",
        "hora_inicio": "17:15",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 8,
      "categoria": "simple",
      "texto": "Charla de ciberseguridad el 30 de septiembre a las 15:00",
      "ground_truth": {
        "titulo": "Charla de ciberseguridad",
        "fecha": "2026-09-30",
        "hora_inicio": "15:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 9,
      "categoria": "con_participantes",
      "texto": "Reunión con Sofía y Martín el 16 de septiembre a las 10:30",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-09-16",
        "hora_inicio": "10:30",
        "duracion_minutos": null,
        "participantes": [
          "Sofía",
          "Martín"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 10,
      "categoria": "con_participantes",
      "texto": "Tutoría con el tutor de práctica el viernes a las 15:00",
      "ground_truth": {
        "titulo": "Tutoría",
        "fecha": "2026-09-04",
        "hora_inicio": "15:00",
        "duracion_minutos": null,
        "participantes": [
          "tutor de práctica"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 11,
      "categoria": "con_participantes",
      "texto": "Asado con los primos el sábado a las 13:00",
      "ground_truth": {
        "titulo": "Asado",
        "fecha": "2026-09-05",
        "hora_inicio": "13:00",
        "duracion_minutos": null,
        "participantes": [
          "los primos"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 12,
      "categoria": "con_participantes",
      "texto": "Consulta con la doctora Rojas el 21 de septiembre a las 8:40",
      "ground_truth": {
        "titulo": "Consulta",
        "fecha": "2026-09-21",
        "hora_inicio": "08:40",
        "duracion_minutos": null,
        "participantes": [
          "doctora Rojas"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 13,
      "categoria": "con_participantes",
      "texto": "Almuerzo con Javiera, Benja e Ignacio el jueves a las 13:30",
      "ground_truth": {
        "titulo": "Almuerzo",
        "fecha": "2026-09-03",
        "hora_inicio": "13:30",
        "duracion_minutos": null,
        "participantes": [
          "Javiera",
          "Benja",
          "Ignacio"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 14,
      "categoria": "con_participantes",
      "texto": "Standup con el equipo de backend el lunes a las 9:15",
      "ground_truth": {
        "titulo": "Standup",
        "fecha": "2026-09-07",
        "hora_inicio": "09:15",
        "duracion_minutos": null,
        "participantes": [
          "equipo de backend"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 15,
      "categoria": "con_participantes",
      "texto": "Call with Priya and Tom on Wednesday at 11am",
      "ground_truth": {
        "titulo": "Call",
        "fecha": "2026-09-02",
        "hora_inicio": "11:00",
        "duracion_minutos": null,
        "participantes": [
          "Priya",
          "Tom"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 16,
      "categoria": "con_participantes",
      "texto": "Reunión de apoderados con los profesores jefes el 24 de septiembre a las 19:00",
      "ground_truth": {
        "titulo": "Reunión de apoderados",
        "fecha": "2026-09-24",
        "hora_inicio": "19:00",
        "duracion_minutos": null,
        "participantes": [
          "los profesores jefes"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 17,
      "categoria": "ambigua",
      "texto": "Kinesiólogo mañana a las 18:30",
      "ground_truth": {
        "titulo": "Kinesiólogo",
        "fecha": "2026-09-02",
        "hora_inicio": "18:30",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 18,
      "categoria": "ambigua",
      "texto": "Revisión de avance pasado mañana a las 11",
      "ground_truth": {
        "titulo": "Revisión de avance",
        "fecha": "2026-09-03",
        "hora_inicio": "11:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 19,
      "categoria": "ambigua",
      "texto": "Retro del sprint el jueves de la próxima semana a las 16:00",
      "ground_truth": {
        "titulo": "Retro del sprint",
        "fecha": "2026-09-10",
        "hora_inicio": "16:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 20,
      "categoria": "ambigua",
      "texto": "Clase de guitarra en tres semanas a las 17:00",
      "ground_truth": {
        "titulo": "Clase de guitarra",
        "fecha": "2026-09-22",
        "hora_inicio": "17:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 21,
      "categoria": "ambigua",
      "texto": "Peluquería el próximo sábado a las 10",
      "ground_truth": {
        "titulo": "Peluquería",
        "fecha": "2026-09-05",
        "hora_inicio": "10:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 22,
      "categoria": "ambigua",
      "texto": "Defensa del proyecto en 10 días a las 9:00",
      "ground_truth": {
        "titulo": "Defensa del proyecto",
        "fecha": "2026-09-11",
        "hora_inicio": "09:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 23,
      "categoria": "ambigua",
      "texto": "Feria el domingo a las 12",
      "ground_truth": {
        "titulo": "Feria",
        "fecha": "2026-09-06",
        "hora_inicio": "12:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 24,
      "categoria": "ambigua",
      "texto": "Reunión de pauta el viernes 18 a las 10",
      "ground_truth": {
        "titulo": "Reunión de pauta",
        "fecha": "2026-09-18",
        "hora_inicio": "10:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 25,
      "categoria": "compleja",
      "texto": "Defensa de práctica el miércoles 16 de septiembre a las 15:00 por 40 minutos con la comisión en el Aula Magna, avísame 20 minutos antes",
      "ground_truth": {
        "titulo": "Defensa de práctica",
        "fecha": "2026-09-16",
        "hora_inicio": "15:00",
        "duracion_minutos": 40,
        "participantes": [
          "comisión"
        ],
        "ubicacion": "Aula Magna",
        "recordatorio_minutos": 20,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 26,
      "categoria": "compleja",
      "texto": "Sesión de estudio con Valentina y Diego el 23/09 a las 17:00 en la biblioteca central, 2 horas y media",
      "ground_truth": {
        "titulo": "Sesión de estudio",
        "fecha": "2026-09-23",
        "hora_inicio": "17:00",
        "duracion_minutos": 150,
        "participantes": [
          "Valentina",
          "Diego"
        ],
        "ubicacion": "biblioteca central",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 27,
      "categoria": "compleja",
      "texto": "Capacitación de Excel el lunes 28 de septiembre de 14 a 17 en el laboratorio 3 con los practicantes",
      "ground_truth": {
        "titulo": "Capacitación de Excel",
        "fecha": "2026-09-28",
        "hora_inicio": "14:00",
        "duracion_minutos": 180,
        "participantes": [
          "los practicantes"
        ],
        "ubicacion": "laboratorio 3",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 28,
      "categoria": "compleja",
      "texto": "Entrevista con la candidata a diseñadora el 1 de octubre a las 11 por Google Meet, media hora, recordatorio 5 minutos antes",
      "ground_truth": {
        "titulo": "Entrevista",
        "fecha": "2026-10-01",
        "hora_inicio": "11:00",
        "duracion_minutos": 30,
        "participantes": [
          "candidata a diseñadora"
        ],
        "ubicacion": "Google Meet",
        "recordatorio_minutos": 5,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 29,
      "categoria": "compleja",
      "texto": "Cena de aniversario el sábado 26 de septiembre a las 21:00 en el restaurante Donde Pepe con Fernanda, 3 horas",
      "ground_truth": {
        "titulo": "Cena de aniversario",
        "fecha": "2026-09-26",
        "hora_inicio": "21:00",
        "duracion_minutos": 180,
        "participantes": [
          "Fernanda"
        ],
        "ubicacion": "restaurante Donde Pepe",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 30,
      "categoria": "compleja",
      "texto": "Sprint planning on Monday September 21st at 9am in Conference Room 4 with the QA team for 90 minutes, remind me 15 minutes before",
      "ground_truth": {
        "titulo": "Sprint planning",
        "fecha": "2026-09-21",
        "hora_inicio": "09:00",
        "duracion_minutos": 90,
        "participantes": [
          "QA team"
        ],
        "ubicacion": "Conference Room 4",
        "recordatorio_minutos": 15,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 31,
      "categoria": "compleja",
      "texto": "Consejo de escuela el jueves 17 de septiembre a las 12:00 en la sala de reuniones del decanato, 1 hora, con los 8 jefes de carrera",
      "ground_truth": {
        "titulo": "Consejo de escuela",
        "fecha": "2026-09-17",
        "hora_inicio": "12:00",
        "duracion_minutos": 60,
        "participantes": [
          "los 8 jefes de carrera"
        ],
        "ubicacion": "sala de reuniones del decanato",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 32,
      "categoria": "compleja",
      "texto": "Operativo de vacunación el 3/10 de 8 a 13 en el gimnasio municipal, avísame una hora antes",
      "ground_truth": {
        "titulo": "Operativo de vacunación",
        "fecha": "2026-10-03",
        "hora_inicio": "08:00",
        "duracion_minutos": 300,
        "participantes": [],
        "ubicacion": "gimnasio municipal",
        "recordatorio_minutos": 60,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 33,
      "categoria": "edge_case",
      "texto": "Llamada con el contador",
      "ground_truth": {
        "titulo": "Llamada",
        "fecha": null,
        "hora_inicio": null,
        "duracion_minutos": null,
        "participantes": [
          "contador"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 34,
      "categoria": "edge_case",
      "texto": "Pilates a las 7:15 am",
      "ground_truth": {
        "titulo": "Pilates",
        "fecha": null,
        "hora_inicio": "07:15",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 35,
      "categoria": "edge_case",
      "texto": "Meeting with the board next Thursday",
      "ground_truth": {
        "titulo": "Meeting",
        "fecha": "2026-09-03",
        "hora_inicio": null,
        "duracion_minutos": null,
        "participantes": [
          "board"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 36,
      "categoria": "edge_case",
      "texto": "Llamada con Lucía el 27 de septiembre a las 8 pm hora de Madrid",
      "ground_truth": {
        "titulo": "Llamada",
        "fecha": "2026-09-27",
        "hora_inicio": "20:00",
        "duracion_minutos": null,
        "participantes": [
          "Lucía"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "Europe/Madrid"
      }
    },
    {
      "id": 37,
      "categoria": "edge_case",
      "texto": "Reunión el 31/09 a las 10",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": null,
        "hora_inicio": "10:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 38,
      "categoria": "edge_case",
      "texto": "Mantención del auto el 5 de octubre",
      "ground_truth": {
        "titulo": "Mantención del auto",
        "fecha": "2026-10-05",
        "hora_inicio": null,
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 39,
      "categoria": "edge_case",
      "texto": "Reunión el 12 de octubre a las 25:00",
      "ground_truth": {
        "titulo": "Reunión",
        "fecha": "2026-10-12",
        "hora_inicio": null,
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 40,
      "categoria": "edge_case",
      "texto": "Doctor's appointment tomorrow at 7:30am",
      "ground_truth": {
        "titulo": "appointment",
        "fecha": "2026-09-02",
        "hora_inicio": "07:30",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 41,
      "categoria": "chat",
      "texto": "oye agéndame hackathon el sábado de 10 a 18 en el DIICC",
      "ground_truth": {
        "titulo": "hackathon",
        "fecha": "2026-09-05",
        "hora_inicio": "10:00",
        "duracion_minutos": 480,
        "participantes": [],
        "ubicacion": "DIICC",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 42,
      "categoria": "chat",
      "texto": "wena, agenda un café con la Cata el miércoles a las 6 en el Starbucks del mall",
      "ground_truth": {
        "titulo": "café",
        "fecha": "2026-09-02",
        "hora_inicio": "18:00",
        "duracion_minutos": null,
        "participantes": [
          "Cata"
        ],
        "ubicacion": "Starbucks del mall",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 43,
      "categoria": "chat",
      "texto": "porfa anota tutoría de cálculo el lunes a las 3, dura 45 min",
      "ground_truth": {
        "titulo": "tutoría de cálculo",
        "fecha": "2026-09-07",
        "hora_inicio": "15:00",
        "duracion_minutos": 45,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 44,
      "categoria": "chat",
      "texto": "mañana a las 8 tengo control con el dentista, recuérdame media hora antes",
      "ground_truth": {
        "titulo": "control",
        "fecha": "2026-09-02",
        "hora_inicio": "08:00",
        "duracion_minutos": null,
        "participantes": [
          "dentista"
        ],
        "ubicacion": null,
        "recordatorio_minutos": 30,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 45,
      "categoria": "chat",
      "texto": "necesito agendar la entrega del informe pa pasado mañana a las 23:59",
      "ground_truth": {
        "titulo": "entrega del informe",
        "fecha": "2026-09-03",
        "hora_inicio": "23:59",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 46,
      "categoria": "chat",
      "texto": "hey can u set up a sync w/ Alex and Jordan on thursday 2pm on Slack huddle",
      "ground_truth": {
        "titulo": "sync",
        "fecha": "2026-09-03",
        "hora_inicio": "14:00",
        "duracion_minutos": null,
        "participantes": [
          "Alex",
          "Jordan"
        ],
        "ubicacion": "Slack huddle",
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 47,
      "categoria": "chat",
      "texto": "la junta de vecinos es el domingo a las 11 en la sede, avisa 1 hora antes xfa",
      "ground_truth": {
        "titulo": "junta de vecinos",
        "fecha": "2026-09-06",
        "hora_inicio": "11:00",
        "duracion_minutos": null,
        "participantes": [],
        "ubicacion": "sede",
        "recordatorio_minutos": 60,
        "zona_horaria": "America/Santiago"
      }
    },
    {
      "id": 48,
      "categoria": "chat",
      "texto": "almorcemos con los del lab el viernes a la 1",
      "ground_truth": {
        "titulo": "Almuerzo",
        "fecha": "2026-09-04",
        "hora_inicio": "13:00",
        "duracion_minutos": null,
        "participantes": [
          "los del lab"
        ],
        "ubicacion": null,
        "recordatorio_minutos": null,
        "zona_horaria": "America/Santiago"
      }
    }
  ]
}

In [ ]:
# @title 2. Ubicar el adapter LoRA (checkpoint-300)
ORIGEN_ADAPTER = "subir archivos"  # @param ["subir archivos", "drive"]
RUTA_DRIVE = "/content/drive/MyDrive/qwen3-4b-semantic-lora/checkpoint-300"  # @param {type:"string"}
import os
from google.colab import files

NECESARIOS = ["adapter_config.json", "adapter_model.safetensors"]

if ORIGEN_ADAPTER == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ADAPTER = RUTA_DRIVE
else:
    ADAPTER = "/content/adapter"
    os.makedirs(ADAPTER, exist_ok=True)
    faltan = [f for f in NECESARIOS if not os.path.isfile(os.path.join(ADAPTER, f))]
    if faltan:
        print("Sube:", ", ".join(faltan), "(de checkpoint-300; el .safetensors pesa ~130 MB)")
        for nombre, datos in files.upload().items():
            # Colab renombra repetidos como "adapter_config (1).json"
            destino = next((n for n in NECESARIOS if nombre.startswith(n.split(".")[0])), nombre)
            with open(os.path.join(ADAPTER, destino), "wb") as f:
                f.write(datos)
            os.remove(nombre)

for f in NECESARIOS:
    assert os.path.isfile(os.path.join(ADAPTER, f)), (
        f"Falta {f} en {ADAPTER}. Con 'drive': crea el acceso directo a qwen3-4b-semantic-lora en Mi unidad.")
print("Adapter:", ADAPTER)

In [ ]:
# @title 3. Cargar Qwen3-4B + LoRA (~8 GB)
import time
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

VERSION = "despues"
MODELO_ID = "Qwen/Qwen3-4B-Instruct-2507"

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)
base = AutoModelForCausalLM.from_pretrained(MODELO_ID, dtype=torch.float16, device_map="auto")
# merge: los pesos quedan fundidos y la inferencia cuesta lo mismo que el modelo base
model = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload().eval()
SEGUNDOS_CARGA = time.perf_counter() - t0
print(f"{MODELO_ID} + LoRA ({ADAPTER}) cargado en {model.device} en {SEGUNDOS_CARGA:.0f} s")

In [ ]:
# @title 4. Extractor (idéntico al del chat)
import contextlib
import json
import re
from datetime import date, datetime
from zoneinfo import ZoneInfo
import resolvedor_temporal as rt

# Identico a PROMPT_SEMANTICO de pipeline/prompts.py del repo: es el prompt con que se entreno el LoRA.
PROMPT_SEMANTICO = """Eres un extractor de datos de calendario.

REGLAS:
1. Devuelve SOLO JSON con estos seis campos: titulo, hora_inicio, duracion_minutos, participantes, ubicacion, recordatorio_minutos.
2. NO calcules ni devuelvas fechas: el campo fecha lo resuelve otro componente del sistema.
3. NO devuelvas zona_horaria ni recurrence: los calcula el sistema con reglas.
4. Si un campo NO se menciona, usa null (o [] para participantes). NUNCA inventes.
5. titulo: nombre breve del evento; elimina hora, participantes y ubicacion, pero conserva sus descriptores (p. ej., "Cita médica", "Reunión de planificación"). Conserva el idioma de la frase.
6. Si la frase es un verbo de encuentro sin nombre de evento (p. ej. "quedamos el viernes", "presentamos el lunes"), usa la etiqueta nominal del evento: "quedamos" -> "Quedada", "presentamos" -> "Presentación", "comemos" -> "Almuerzo".
7. hora_inicio: HH:MM 24h. Si no hay = null.
8. duracion_minutos: entero (1 hora=60). Si no hay = null.
9. participantes: nombres tal como se mencionan, PERO quitando el articulo inicial singular ("el"/"la"/"the"): "con el cliente" -> ["cliente"], "con la Dani" -> ["Dani"], "with the team" -> ["team"]. El articulo plural o total SI se conserva: "con los clientes" -> ["los clientes"], "con todo el equipo" -> ["todo el equipo"], "con los 5 directores" -> ["los 5 directores"]. Excepcion: "el profe" se conserva entero -> ["el profe"]. Si no hay = [].
10. ubicacion: donde se realiza. Si no se menciona = null.
11. recordatorio_minutos: entero. Si no se menciona = null."""

ZONA_DEFECTO = "America/Santiago"
ZONAS = {"madrid": "Europe/Madrid", "argentina": "America/Argentina/Buenos_Aires", "santiago": "America/Santiago"}
DIAS_EN = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
_DIA_A_EN = {nombre: DIAS_EN[n - 1] for nombre, n in rt._DIAS.items()}
_DIAS_RE = "|".join(sorted(_DIA_A_EN, key=len, reverse=True))
_NUM = {"dos": 2, "tres": 3, "cuatro": 4, "cinco": 5, "seis": 6, "diez": 10,
        "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "ten": 10, "other": 2}
_NUM_RE = r"\d+|" + "|".join(_NUM)


def hoy_santiago():
    return datetime.now(ZoneInfo("America/Santiago")).date()


def _generar(mensajes, con_adapter=True, max_new_tokens=256):
    """Decodificacion greedy (determinista). con_adapter=False -> Qwen3-4B base, sin fine-tuning."""
    try:
        prompt = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True,
                                               enable_thinking=False)
    except TypeError:
        prompt = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    sin_adapter = contextlib.nullcontext() if con_adapter else model.disable_adapter()
    with torch.no_grad(), sin_adapter:
        salida = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                temperature=None, top_p=None, top_k=None,
                                pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(salida[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()


def _extraer_json(respuesta):
    """Primer objeto JSON valido de la respuesta, o None."""
    decoder = json.JSONDecoder()
    for m in re.finditer(r"\{", respuesta):
        try:
            obj, _ = decoder.raw_decode(respuesta[m.start():])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue
    return None


# --- normalizacion de los 6 campos del modelo (como pipeline/normalizacion.py del repo) ---

def _hora(valor):
    if not isinstance(valor, str):
        return None
    v = valor.strip()
    m = re.fullmatch(r"(\d{2}):(\d{2})", v)
    if m:
        # ya viene en HH:MM 24h: no pasarlo por la convencion "1..7 = tarde" del resolvedor
        # (el repo lo hace y convierte 07:15 en 19:15)
        return v if int(m.group(1)) <= 23 and int(m.group(2)) <= 59 else None
    return rt.resolver_hora_expr(v)


def _entero(valor):
    if isinstance(valor, bool):
        return None
    if isinstance(valor, (int, float)):
        return int(valor)
    if isinstance(valor, str):
        m = re.search(r"\d+", valor)
        return int(m.group()) if m else None
    return None


def _participantes(valor):
    if not isinstance(valor, list):
        return []
    return [x.strip() for x in valor
            if isinstance(x, str) and x.strip() and not x.strip().startswith("<") and not x.strip().isdigit()]


def _ubicacion(valor, texto):
    if isinstance(valor, str) and valor.strip():
        return valor.strip()
    t = texto.lower()
    return next((e for e in ["virtual", "videollamada", "google meet", "zoom", "teams"] if e in t), None)


def _zona(texto):
    t = texto.lower()
    return next((z for clave, z in ZONAS.items() if clave in t), ZONA_DEFECTO)


# --- reglas deterministas: recurrencia y "hoy" (el resolvedor del repo no los cubre) ---

def _quitar_hasta(texto, ref):
    """'hasta el 30 de octubre' -> (fecha de termino, texto sin esa expresion)."""
    m = re.search(r"\b(hasta|until)\b", texto, re.I)
    if not m:
        return None, texto
    resto = texto[m.end():m.end() + 30]
    a = rt.analizar_temporal(resto, ref)
    if not a["fecha"] or not a["fecha_expr"]:
        return None, texto
    fin = m.end() + resto.find(a["fecha_expr"]) + len(a["fecha_expr"])
    return a["fecha"], texto[:m.start()] + texto[fin:]


def detectar_recurrencia(texto, ref):
    """Devuelve (recurrencia del contrato o None, texto sin la expresion 'hasta ...')."""
    t = rt._norm(texto)
    dias = []
    for m in re.finditer(r"\b(?:todos\s+los|todas\s+las|cada|every|each|los)\s+((?:%s)s?(?:\s*(?:,|y|e|and)\s*(?:%s)s?)*)"
                         % (_DIAS_RE, _DIAS_RE), t):
        dias += [_DIA_A_EN[d] for d in re.findall(_DIAS_RE, m.group(1))]
    dias += [d for d in re.findall(r"\b(%s)s\b" % "|".join(DIAS_EN), t)]          # "mondays"
    dias = sorted(set(dias), key=DIAS_EN.index)

    frecuencia, intervalo = None, 1
    if dias:
        frecuencia = "weekly"
    elif re.search(r"\b(todos\s+los\s+dias|cada\s+dia|a\s+diario|diari[oa]s?|diariamente|every\s+day|daily)\b", t):
        frecuencia = "daily"
    elif re.search(r"\b(semanal(mente)?|cada\s+semana|todas\s+las\s+semanas|every\s+week|weekly)\b", t):
        frecuencia = "weekly"
    elif re.search(r"\b(mensual(mente)?|cada\s+mes|todos\s+los\s+meses|every\s+month|monthly)\b", t):
        frecuencia = "monthly"
    elif re.search(r"\bquincenal(mente)?\b", t):
        frecuencia, intervalo = "weekly", 2
    m = re.search(r"\b(?:cada|every)\s+(%s)\s+(dias?|days?|semanas?|weeks?|mes(?:es)?|months?)\b" % _NUM_RE, t)
    if m:
        intervalo = int(m.group(1)) if m.group(1).isdigit() else _NUM[m.group(1)]
        unidad = m.group(2)
        frecuencia = "daily" if unidad.startswith(("dia", "day")) else \
                     "monthly" if unidad.startswith(("mes", "month")) else "weekly"
    if frecuencia is None:
        return None, texto

    hasta, texto_sin_hasta = _quitar_hasta(texto, ref)
    veces = re.search(r"\b(%s)\s+(?:veces|times|sesiones|ocurrencias)\b" % _NUM_RE, t)
    veces = (int(veces.group(1)) if veces.group(1).isdigit() else _NUM[veces.group(1)]) if veces else None
    return {"frecuencia": frecuencia, "intervalo": intervalo,
            "dias": dias or None if frecuencia == "weekly" else None,
            "hasta": hasta, "veces": None if hasta else veces}, texto_sin_hasta


def resolver_fecha(texto, ref, recurrencia):
    fecha = rt.resolver_fecha(texto, ref)
    if fecha:
        return fecha
    if re.search(r"\b(hoy|today|tonight|esta\s+(tarde|noche|manana))\b", rt._norm(texto)):
        return ref
    if recurrencia and recurrencia.get("dias"):
        # "cada lunes a las 10": la primera ocurrencia es el proximo dia de la lista
        base = datetime.strptime(ref, "%Y-%m-%d").date()
        return min(rt._proximo_weekday(base, DIAS_EN.index(d) + 1) for d in recurrencia["dias"]).isoformat()
    return None


def extraer(texto, ref=None):
    """Mensaje -> (evento de 9 campos, info). info: salida cruda del LoRA, segundos y si hubo error."""
    ref = ref or hoy_santiago().isoformat()
    t0 = time.perf_counter()
    crudo = _generar([{"role": "system", "content": PROMPT_SEMANTICO},
                      {"role": "user", "content": 'Frase: "%s"' % texto}])
    segundos = time.perf_counter() - t0
    obj = _extraer_json(crudo)
    recurrencia, texto_fecha = detectar_recurrencia(texto, ref)
    evento = {
        "titulo": (obj or {}).get("titulo") or None,
        "fecha": resolver_fecha(texto_fecha, ref, recurrencia),
        "hora_inicio": _hora((obj or {}).get("hora_inicio")),
        "duracion_minutos": _entero((obj or {}).get("duracion_minutos")),
        "participantes": _participantes((obj or {}).get("participantes")),
        "ubicacion": _ubicacion((obj or {}).get("ubicacion"), texto),
        "recordatorio_minutos": _entero((obj or {}).get("recordatorio_minutos")),
        "zona_horaria": _zona(texto),
        "recurrencia": recurrencia,
    }
    return evento, {"crudo": crudo, "segundos": segundos, "error": obj is None}


def baseline(texto):
    """Mismo prompt directo del chat original, con Qwen3-4B base (adapter desactivado)."""
    return _generar([{"role": "user", "content": f'Agenda esta reunion en el calendario: "{texto}"'}],
                    con_adapter=False)


print("Extractor listo")

BENCHMARKS = ['frases.json', 'benchmark_ft.json']

In [ ]:
# @title 5. Correr los benchmarks (~10 min)
import benchmark_ft as bft

torch.cuda.reset_peak_memory_stats()
ARCHIVOS = []
for bench in BENCHMARKS:
    print(f"\n=== {bench} ===")
    resultados = bft.correr(bench, extraer)
    ruta = f"resultados_{VERSION}_{bench.replace('.json', '')}.json"
    bft.guardar(resultados, ruta)
    ARCHIVOS.append(ruta)
    print("Guardado:", ruta)

props = torch.cuda.get_device_properties(0)
HARDWARE = {
    "version": VERSION, "modelo": MODELO_ID, "adapter": globals().get("ADAPTER"),
    "gpu": torch.cuda.get_device_name(0), "vram_total_gb": round(props.total_memory / 2**30, 1),
    "vram_max_usada_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    "dtype": str(next(model.parameters()).dtype), "decodificacion": "greedy, max_new_tokens=256",
    "segundos_carga": round(SEGUNDOS_CARGA),
    "torch": torch.__version__, "transformers": transformers.__version__,
}
bft.guardar(HARDWARE, f"hardware_{VERSION}.json")
ARCHIVOS.append(f"hardware_{VERSION}.json")
print(json.dumps(HARDWARE, indent=2))

In [ ]:
# @title 6. Pruebas automáticas: PASA / FALLA por frase
REPORTES = {}
for bench in BENCHMARKS:
    with open(f"resultados_{VERSION}_{bench.replace('.json', '')}.json", encoding="utf-8") as f:
        resultados = json.load(f)
    metricas, evaluados = bft.probar(bench, resultados, f"{VERSION} - {bench}")
    ruta = f"reporte_{VERSION}_{bench.replace('.json', '')}.json"
    bft.guardar({"metricas": metricas, "detalles_por_frase": evaluados}, ruta)
    ARCHIVOS.append(ruta)
    REPORTES[bench] = metricas

print("\nRESUMEN", VERSION)
for bench, m in REPORTES.items():
    print(f"  {bench:20s} {m['correctos_totales']}/{m['total_frases']} frases correctas ({m['precision_total']}%)")

In [ ]:
# @title 7. Descargar resultados_despues.zip
import zipfile
from google.colab import files

with zipfile.ZipFile("resultados_despues.zip", "w") as z:
    for ruta in ARCHIVOS:
        z.write(ruta)
print("Contenido:", ARCHIVOS)
files.download("resultados_despues.zip")

In [ ]:
# @title 8. Comparar antes vs después (sube resultados_antes.zip)
import os
import zipfile
from google.colab import files

if not os.path.exists("resultados_antes.zip"):
    print("Sube resultados_antes.zip (lo descargó benchmark_antes.ipynb):")
    for nombre in files.upload():
        if nombre != "resultados_antes.zip":
            os.replace(nombre, "resultados_antes.zip")  # Colab renombra repetidos: "resultados_antes (1).zip"
with zipfile.ZipFile("resultados_antes.zip") as z:
    z.extractall(".")

COMPARACION = []
for bench in BENCHMARKS:
    nombre = bench.replace(".json", "")
    with open(f"resultados_antes_{nombre}.json", encoding="utf-8") as f:
        antes = json.load(f)
    with open(f"resultados_despues_{nombre}.json", encoding="utf-8") as f:
        despues = json.load(f)
    resumen = bft.comparar(bench, antes, despues)
    bft.guardar(resumen, f"comparacion_{nombre}.json")
    COMPARACION.append(f"comparacion_{nombre}.json")

with zipfile.ZipFile("comparacion.zip", "w") as z:
    for ruta in COMPARACION + ["hardware_antes.json", "hardware_despues.json"]:
        if os.path.exists(ruta):
            z.write(ruta)
files.download("comparacion.zip")